In [ ]:
from math import ceil
from typing import cast

import torch
from torch import nn
from torch.utils.data import DataLoader

from fun.data.ct_dataset import CTPostProcessDataset
from fun.data.ellipses_dataset import EllipsesDataset
from fun.models.custom_unet import CustomUNet
from fun.utils.fno_utils import gen_from_Conv2d

In [ ]:
weights = torch.load("../runs/UNet-64x64-11/weights/best-all.pt", map_location="cpu")
model = CustomUNet(1, 1, nonresize_convs_per_block=1)
model.load_state_dict(weights)

j = 1
for block in model._down_blocks:  # pyright: ignore [reportPrivateUsage]
    for i, layer in enumerate(cast(nn.ModuleList, block)):
        if isinstance(layer, nn.Conv2d):
            cast(nn.ModuleList, block)[i] = gen_from_Conv2d(layer, 256//2**j, 256//2**j)
    j += 1
for i, layer in enumerate(model._central_block):  # pyright: ignore [reportPrivateUsage]
    if isinstance(layer, nn.Conv2d):
        model._central_block[i] = gen_from_Conv2d(layer, 256//2**len(model._down_blocks), 256//2**len(model._down_blocks))  # pyright: ignore [reportPrivateUsage]
j = len(model._down_blocks) - 1
for block in model._up_blocks:  # pyright: ignore [reportPrivateUsage]
    for i, layer in enumerate(cast(nn.ModuleList, block)):
        if isinstance(layer, nn.Conv2d):
            cast(nn.ModuleList, block)[i] = gen_from_Conv2d(layer, 256//2**j, 256//2**j)
    j -= 1

In [15]:
test_datasets = {
    f"{r}x{r}": CTPostProcessDataset(
        torch.utils.data.Subset(EllipsesDataset.from_file("../data/test.h5"), range(200)),
        angles=torch.linspace(0.0, torch.pi * 0.75, ceil(4 * r * 0.75)),
        pos_count=2 * r,
        target_shape=(r, r),
        noise_type="gaussian",
        noise_level=0.0,
    )
    for r in [x for i in range(4, 10) for x in [2**i, 2**i + 2 ** (i - 2) + 1, 2**i + 2 ** (i - 1), 2 ** (i + 1) - 2 ** (i - 2) - 1, 2**i + 1]]
}
test_dataloaders = {
    name: DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )
    for name, dataset in test_datasets.items()
}

In [ ]:
DEVICE = "cpu" if not torch.cuda.is_available() else "cuda"

loss_function = nn.MSELoss()
model.to(DEVICE)
for name, dataloader in test_dataloaders.items():
    try:
        loss = 0.0
        for batch in dataloader:
            input_ = batch["input"].to(DEVICE)
            target = batch["target"].to(DEVICE)
            prediction = model(input_)
            loss += loss_function(prediction, target).item()
        loss /= len(dataloader)
    except Exception as e:
        print(f"Error testing {name} dataset: {e}")
        continue
    print(f"Finished testing {name} dataset: {loss}")

Finished testing 16x16 dataset: 895.8828037806919
Error testing 21x21 dataset: Input width is not divisible by 16, got 21 (21 / 16 = 1.3125).
Error testing 24x24 dataset: Input width is not divisible by 16, got 24 (24 / 16 = 1.5).
Error testing 27x27 dataset: Input width is not divisible by 16, got 27 (27 / 16 = 1.6875).
Error testing 17x17 dataset: Input width is not divisible by 16, got 17 (17 / 16 = 1.0625).
Finished testing 32x32 dataset: 620.1540788922991
Error testing 41x41 dataset: Input width is not divisible by 16, got 41 (41 / 16 = 2.5625).
Finished testing 48x48 dataset: 471.7549482073103
Error testing 55x55 dataset: Input width is not divisible by 16, got 55 (55 / 16 = 3.4375).
Error testing 33x33 dataset: Input width is not divisible by 16, got 33 (33 / 16 = 2.0625).
Finished testing 64x64 dataset: 378.664803641183
Error testing 81x81 dataset: Input width is not divisible by 16, got 81 (81 / 16 = 5.0625).
Finished testing 96x96 dataset: 286.2153974260603
Error testing 111x